### Auto-encoder is a deep learning alternative to PCA

Technically we de-construct and re-construct a dataset via the autoencoder. Thus having less variables and losing the most noisiest/outlier rows from the data in the process.

Remember: PCA is a linear algorithm, so if data contains many non-linear relationships, it might not be optimal. In addition to auto-encoders, you might also consider Kernel PCA, UMA or Isomap.

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np 

# load the data
df = pd.read_csv("winequality-red.csv")

# X/y -split
X = df.drop("quality", axis=1)
y = df['quality']

# scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

**Version 1: Use autoencoders to reduce dimensionality (amount of variables)**

In [16]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# define the input dimension
# in this case 12 variables - target variable = 11
input_dim = X_train.shape[1]

# basically => we are reducing from original amount of variables
# into => 8 variables
encoding_dim = 10

input_layer = Input(shape=(input_dim,))

# encoder/decooder => autoencoder
# this is "the bottleneck" that reduces the amount of variables
encoded = Dense(encoding_dim, activation='relu')(input_layer)

# this version with only encode -> decode is called a Shallow AutoEncoder
# if you add more hidden layers => it's called => Deep AutoEncoder
# you can also use linear if your target is continuous => regression
decoded = Dense(input_dim, activation='sigmoid')(encoded)

# combine the layers above into a complete autoencoder
autoencoder = Model(inputs=input_layer, outputs=decoded)

# encoder model that reduces dimensionality
encoder = Model(inputs=input_layer, outputs=encoded)

# compile and train 
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(X_train, X_train, epochs=50, batch_size = 32, shuffle=True, validation_split=0.2)

# apply the dimensionality reduction
X_train_encoded = encoder.predict(X_train)
X_test_encoded = encoder.predict(X_test)

Epoch 1/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.2771 - val_loss: 1.0707
Epoch 2/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.1006 - val_loss: 0.9595
Epoch 3/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9822 - val_loss: 0.8759
Epoch 4/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8901 - val_loss: 0.8058
Epoch 5/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8137 - val_loss: 0.7442
Epoch 6/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7462 - val_loss: 0.6881
Epoch 7/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6852 - val_loss: 0.6341
Epoch 8/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6278 - val_loss: 0.5833
Epoch 9/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5754 - val_loss: 0.5349
Epoch 10/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5283 - val_loss: 0.4938
Epoch 11/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4874 - val_loss: 0.4578
Epoch 12/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4516 - val_lo

In [17]:
# create a pandas DataFrame for easier inspection
num_components = X_train_encoded.shape[1]

# create a DataFrame
encoded_df = pd.DataFrame(X_train_encoded, columns=[f"Component {i + 1}" for i in range(num_components)])

# also add the target back
encoded_df['quality'] = y_train

# now we have similar components as in PCA
# but we have no idea of the loadings
# since we are using an autoencoder
# you can use SHAP/LIME or other tools 
# for estimating what affects what

# due to the lack of metrics, if the point is to optimize
# the dataset (reduce variable or noise etc.)
# consider using PCA, UMAP or Kernel PCA, based on 
# the type of dataset linear / non-linear
encoded_df

,Component 1,Component 2,Component 3,Component 4,Component 5,Component 6,Component 7,Component 8,Component 9,Component 10,quality
0,0.386888,0.866374,2.381592,1.008679,0.508891,1.164263,1.317744,0.000000,0.414387,1.458286,5.0
1,1.404734,1.296155,0.182978,2.514520,0.782003,2.124517,3.196532,0.409785,2.717834,0.000000,5.0
2,1.570811,0.000000,0.822369,0.250725,0.610258,0.841252,0.611065,1.521811,0.000000,1.982736,5.0
3,0.213796,0.000000,0.794518,0.885001,1.277658,0.777897,0.761125,1.354792,0.000000,0.224347,6.0
4,0.000000,1.914471,1.739525,0.935572,2.309391,0.688348,0.345669,0.639302,0.120609,1.571721,5.0
...,...,...,...,...,...,...,...,...,...,...,...
1274,0.000000,0.000000,0.162347,0.432378,0.349006,0.033261,0.000000,1.178215,0.000000,1.094924,6.0
1275,0.457459,0.343422,1.039767,2.022581,0.563677,0.668472,0.778239,0.617407,0.625621,0.544424,6.0
1276,0.000000,0.333026,1.035320,0.883752,1.322435,0.000000,1.724702,0.701929,0.586519,0.000000,4.0
1277,1.312511,0.000000,0.000000,0.014376,0.687111,0.915119,0.357012,0.000000,1.800531,0.814316,6.0


**Version 2: remove noise, outliers, overlap etc. everything that is not "main trend"**

In [18]:
# calculate reconstruction errors
reconstructed = autoencoder.predict(X_train)
reconstruction_error = np.mean(np.square(X_train - reconstructed), axis=1)

reconstructed_test = autoencoder.predict(X_test)
test_reconstruction_error = np.mean(np.square(X_test - reconstructed_test), axis=1)
print("Train error mean:", np.mean(reconstruction_error))
print("Test error mean:", np.mean(test_reconstruction_error))

# anomalies threshold, this case top 5% is considered noise etc.
threshold = np.percentile(reconstruction_error, 95)

# get anomalies / noise etc.
anomalies = reconstruction_error > threshold
anomaly_indices = np.where(anomalies)[0]

# get the noisy/anomaly rows
# anomaly_mask = df.index.isin(anomaly_indices)
train_indices = X_train.shape[0]
original_train_indices = y_train.index  # keep original indices from the split
anomaly_indices = original_train_indices[anomalies]
df.loc[anomaly_indices]



40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Train error mean: 0.0646795693079575
Test error mean: 0.06499816207314277


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
354,6.1,0.21,0.40,1.4,0.066,40.5,165.0,0.99120,3.25,0.59,11.9,6
339,12.5,0.28,0.54,2.3,0.082,12.0,29.0,0.99970,3.11,1.36,9.8,7
433,12.3,0.39,0.63,2.3,0.091,6.0,18.0,1.00040,3.16,0.49,9.5,5
582,11.7,0.49,0.49,2.2,0.083,5.0,15.0,1.00000,3.19,0.43,9.2,5
86,8.6,0.49,0.28,1.9,0.110,20.0,136.0,0.99720,2.93,1.95,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
91,8.6,0.49,0.28,1.9,0.110,20.0,136.0,0.99720,2.93,1.95,9.9,6
564,13.0,0.47,0.49,4.3,0.085,6.0,47.0,1.00210,3.30,0.68,12.7,6
13,7.8,0.61,0.29,1.6,0.114,9.0,29.0,0.99740,3.26,1.56,9.1,5
276,6.9,0.54,0.04,3.0,0.077,7.0,27.0,0.99870,3.69,0.91,9.4,6
